# 00 · Set up the project and download every dataset

Run in Google Colab signed in as **san60lamichhane@gmail.com**. Top to bottom, this notebook:

1. mounts Google Drive
2. puts the code in `MyDrive/PhD-AI-Cyber/code/phd-ai-threat-detection` (from GitHub, or from the zip you uploaded)
3. builds the full folder structure in the repo and in Drive (`setup_project.py`)
4. downloads every dataset from its official source into `MyDrive/PhD-AI-Cyber/data/raw/`, with SHA-256 checksums
5. saves the checksum manifests to GitHub

Safe to re-run: nothing is overwritten, finished files are skipped, broken downloads resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, zipfile
MYDRIVE    = '/content/drive/MyDrive'
DRIVE_ROOT = f'{MYDRIVE}/PhD-AI-Cyber'
REPO_NAME  = 'phd-ai-threat-detection'
CODE_DIR   = f'{DRIVE_ROOT}/code'
REPO_DIR   = f'{CODE_DIR}/{REPO_NAME}'
pathlib.Path(CODE_DIR).mkdir(parents=True, exist_ok=True)

## Get the code

**Option A (GitHub, preferred).** Add a GitHub token as `GITHUB_TOKEN` in Colab's 🔑 Secrets panel
(fine-grained token for `sandesh20lamichhane/phd-ai-threat-detection`, *Contents: read and write*).

**Option B (no GitHub yet).** Upload `phd-ai-threat-detection.zip` into `MyDrive/PhD-AI-Cyber/code/`.
The cell below unzips it. Switch to GitHub later by renaming that folder and re-running.

In [ ]:
from google.colab import userdata
try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/{REPO_NAME}.git'
ZIP = pathlib.Path(CODE_DIR, f'{REPO_NAME}.zip')

if pathlib.Path(REPO_DIR, '.git').exists():
    if TOKEN:
        !git -C "$REPO_DIR" pull --ff-only "$REMOTE" main
    else:
        print('Using the code in Drive; add GITHUB_TOKEN to sync with GitHub.')
elif pathlib.Path(REPO_DIR, 'setup_project.py').exists():
    print('Using the code already in Drive (not a git clone).')
elif TOKEN:
    !git clone "$REMOTE" "$REPO_DIR"
    !git -C "$REPO_DIR" remote set-url origin https://github.com/sandesh20lamichhane/$REPO_NAME.git
elif ZIP.exists():
    zipfile.ZipFile(ZIP).extractall(CODE_DIR)
    print('Unzipped', ZIP.name)
else:
    raise SystemExit('No code found: add GITHUB_TOKEN to Colab secrets, or upload the zip to ' + CODE_DIR)

if pathlib.Path(REPO_DIR, '.git').exists():
    !git -C "$REPO_DIR" config user.name "Sandesh Lamichhane"
    !git -C "$REPO_DIR" config user.email "san30lamichhane@gmail.com"
%cd $REPO_DIR
!pip -q install -r requirements.txt

## Build the folder structure (repo + Drive) and check the code

In [ ]:
!python setup_project.py --drive-root "$DRIVE_ROOT"
!python setup_project.py --tree
!python -m pytest -q tests

## Links that need registration (fill in once)

Three providers put their files behind a short form. Fill each form in your browser, copy the link you land on, and paste it below.
Leave a value empty until you have it; those datasets are simply skipped for now.

| Key | Where to register | What to paste |
|---|---|---|
| `ciciot2023_csv_folder` | https://www.unb.ca/cic/datasets/iotdataset-2023.html | the URL of the **CSV** folder listing (not PCAP) |
| `ciciomt2024_folder` | https://www.unb.ca/cic/datasets/iomt-dataset-2024.html | the URL of the **WiFi_and_MQTT/attacks/** folder listing |
| `lanl_cyber1_urls` | https://csr.lanl.gov/data/cyber1/ | the 5 file links: auth, proc, flows, dns, redteam (.txt.gz) |

UNSW-NB15 and ToN_IoT live in UNSW SharePoint folders that can't be scripted. Download the CSV folders listed in
`config/datasets.yaml` and drag them into `MyDrive/PhD-AI-Cyber/data/raw/unsw_nb15/` and `.../ton_iot/`; the next run checksums them.

In [ ]:
import yaml
GATED = {
    'ciciot2023_csv_folder': '',
    'ciciomt2024_folder': '',
    'lanl_cyber1_urls': [
        # 'https://.../auth.txt.gz',
    ],
}
path = pathlib.Path(DRIVE_ROOT, 'config', 'gated_urls.yaml')
saved = yaml.safe_load(path.read_text()) if path.exists() else {}
saved = saved or {}
saved.update({k: v for k, v in GATED.items() if v})   # never blank out a link saved earlier
path.write_text(yaml.safe_dump(saved))
print('Saved links for:', sorted(saved) or 'none yet')

## Optional: Kaggle fallback

Only used for UNSW-NB15 if you'd rather not do the SharePoint step. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab secrets.
Official files are preferred for the thesis; the manifest records which source was used.

In [ ]:
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    !pip -q install kaggle
    print('Kaggle fallback enabled')
except Exception:
    print('No Kaggle secrets: fallback off (that is fine)')

## 1. Plan: list every file and its size, download nothing

In [ ]:
!python -m src.acquire.fetch --plan

## 2. Download

Downloads every enabled dataset that has what it needs. Stops a dataset before it would push raw data past 190 GB.
To do one dataset only: `!python -m src.acquire.fetch --only nsl_kdd`

In [ ]:
!python -m src.acquire.fetch

## 3. Verify checksums of everything in Drive

In [ ]:
!python -m src.acquire.fetch --verify

## 4. Save the manifests (results) to GitHub

In [ ]:
if pathlib.Path(REPO_DIR, '.git').exists() and TOKEN:
    !git add results/data_manifest
    !git commit -m "Update dataset manifests" || echo "nothing new to commit"
    !git pull --rebase "$REMOTE" main
    !git push "$REMOTE" HEAD:main
else:
    print('No GitHub token yet: manifests are saved in Drive under results/data_manifest/ and data/raw/*/MANIFEST.json')
